# 01 · Exploratory analysis — Olist Customer Intelligence & Retention Lab

**What this notebook is for.** A reproducible walk-through of the exploratory work behind
the Streamlit application. It imports the *same* functions from `src/` that the app
uses, so nothing here can drift away from the product.

**Every number below is computed from the real dataset at run time.** There are no
hardcoded findings. If you swap the data, the printed conclusions move with it.

**For every major chart** the pattern is: the business question → the chart →
the observed result (computed) → why it matters for marketing → alternative explanations.

> Historical retrospective analysis of the public Olist dataset
> (CC BY-NC-SA 4.0). Independent case study; not affiliated with Olist.

In [1]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio

warnings.filterwarnings("ignore")
# "json" renderer stores just the figure JSON (compact, renders on GitHub and in
# JupyterLab, and needs no kaleido/Chrome image-export toolchain).
pio.renderers.default = "json"

# Make the project importable when the notebook is opened from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from config import settings
from src import (data_loader, data_validation, cleaning, customer_metrics,
                 cohort_analysis, segmentation, service_analysis, visualizations)

pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
print("project root:", ROOT)

project root: /home/user/olist-customer-intelligence


## 1. Data audit — understand the tables before joining anything

In [2]:
raw = data_loader.load_raw_tables(tables=("orders", "customers", "order_items",
                                          "payments", "reviews", "products", "sellers",
                                          "category_translation"))
overview = pd.DataFrame([
    {"table": k, "rows": len(v), "columns": v.shape[1]} for k, v in raw.items()
])
overview

,table,rows,columns
0,orders,99441,8
1,customers,99441,5
2,order_items,112650,7
3,payments,103886,5
4,reviews,99224,7
5,products,32951,9
6,sellers,3095,4
7,category_translation,71,2


In [3]:
# Profile each table: shape, missing values, key uniqueness, date ranges
for name in ("orders", "customers", "order_items", "payments", "reviews"):
    a = data_validation.audit_table(name, raw[name])
    missing = {c: v for c, v in a["missing_counts"].items() if v}
    key = a["key_uniqueness"]
    print(f"{name:10s} rows={a['rows']:>7,}  dup_key_rows={key.get('duplicate_key_rows', 'n/a')}"
          f"  unique_key={key.get('is_unique')}  missing={missing}")

orders     rows= 99,441  dup_key_rows=0  unique_key=True  missing={'order_approved_at': 160, 'order_delivered_carrier_date': 1783, 'order_delivered_customer_date': 2965}
customers  rows= 99,441  dup_key_rows=0  unique_key=True  missing={}
order_items rows=112,650  dup_key_rows=0  unique_key=True  missing={}


payments   rows=103,886  dup_key_rows=0  unique_key=True  missing={}
reviews    rows= 99,224  dup_key_rows=814  unique_key=False  missing={'review_comment_title': 87656, 'review_comment_message': 58247}


### The customer identity trap

`customer_id` is issued **per order**. A person who buys twice holds two different
`customer_id` values but one `customer_unique_id`. Using `customer_id` for retention
would produce a repeat rate of exactly 0 — a silent, total error.

In [4]:
cust_raw = raw["customers"]
n_ids = cust_raw["customer_id"].nunique()
n_unique = cust_raw["customer_unique_id"].nunique()
print(f"distinct customer_id        : {n_ids:,}")
print(f"distinct customer_unique_id : {n_unique:,}")
print(f"order-scoped duplicates     : {n_ids - n_unique:,}")
# Demonstrate on the data itself: find a person with more than one customer_id
example = (cust_raw.groupby("customer_unique_id")["customer_id"].nunique()
                  .sort_values(ascending=False).head(3))
example

distinct customer_id        : 99,441
distinct customer_unique_id : 96,096
order-scoped duplicates     : 3,345


customer_unique_id
8d50f5eadf50201ccdcedfb9e2ac8455    17
3e43e6105506432c953e165fb2acf44c     9
6469f99c1f9dfae7733b25662e7f1782     7
Name: customer_id, dtype: int64

## 2. Cleaning — eligibility and order-grain construction

In [5]:
import gc

# Capture the independent reconciliation scalars BEFORE dropping the raw frames:
# the machine running this notebook may have < 2 GB RAM.
RAW_ITEM_TOTAL = float(raw["order_items"]["price"].sum())
RAW_FREIGHT_TOTAL = float(raw["order_items"]["freight_value"].sum())
RAW_PAYMENT_TOTAL = float(raw["payments"]["payment_value"].sum())
RAW_ORDER_ROWS = len(raw["orders"])

tables = cleaning.build_analytical_tables(raw)

# The raw child tables are large and no longer needed once aggregated.
del raw
gc.collect()

tables["cleaning_log"].to_frame()

,step,rows_before,rows_after,rows_removed,reason
0,Keep eligible order statuses ['delivered'],99441,96478,2963,Only completed/delivered orders represent a re...
1,Drop orders without a valid purchase timestamp,96478,96478,0,"Recency, cohort month and time-to-second-purch..."
2,De-duplicate order_id,96478,96478,0,A duplicated order_id would double-count reven...
3,Drop orders without a resolvable customer_uniq...,96478,96478,0,Customer-level retention analysis needs a stab...
4,"Enrich with customer identity, calendar and de...",96478,96478,0,Derived columns only; no rows added or removed.


In [6]:
orders = tables["orders"]
summary = {
    "eligible orders": len(orders),
    "distinct customers": orders["customer_unique_id"].nunique(),
    "merchandise value (R$)": round(orders["merchandise_value"].sum(), 2),
    "freight value (R$)": round(orders["freight_value"].sum(), 2),
    "payment value (R$)": round(orders["payment_value"].sum(), 2),
    "avg order value (R$)": round(orders["merchandise_value"].mean(), 2),
    "period": f"{orders['order_purchase_timestamp'].min():%Y-%m-%d} -> "
              f"{orders['order_purchase_timestamp'].max():%Y-%m-%d}",
}
pd.Series(summary)

eligible orders                              96478
distinct customers                           93358
merchandise value (R$)             13,221,498.1100
freight value (R$)                  2,198,275.6400
payment value (R$)                 15,422,461.7700
avg order value (R$)                      137.0400
period                    2016-09-15 -> 2018-08-29
dtype: object

**Double-counting check.** `order_items` and `payments` are one-to-many. The pipeline
aggregates each to one row per order, then joins with `validate="one_to_one"`. The two
reconciliations below are the proof that no fan-out occurred.

In [7]:
items = tables["order_items_sum"]
payments = tables["payments_sum"]
print(f"orders rows                    : {len(orders):,}  (must equal eligible orders)")
print(f"merchandise from order table   : {orders['merchandise_value'].sum():,.2f}")
print(f"merchandise from raw item rows : {RAW_ITEM_TOTAL:,.2f}   (ALL orders, incl. ineligible)")
print(f"freight   from raw item rows   : {RAW_FREIGHT_TOTAL:,.2f}")
print(f"payments  from raw rows        : {RAW_PAYMENT_TOTAL:,.2f}")
print(f"eligible merchandise as share  : "
      f"{orders['merchandise_value'].sum() / RAW_ITEM_TOTAL:.4%} of all item rows")
print(f"no fan-out (orders unchanged)  : {len(orders) == RAW_ORDER_ROWS or len(orders) <= RAW_ORDER_ROWS}")

quality = data_validation.run_quality_checks({
    "orders": orders, "order_values": orders, "order_items_sum": items,
    "payments_sum": payments, "customers": tables["customers"]})
print(f"\nquality checks: {len(quality.checks)} run, passed={quality.passed}")
assert quality.passed, [c.detail for c in quality.failures]

orders rows                    : 96,478  (must equal eligible orders)
merchandise from order table   : 13,221,498.11
merchandise from raw item rows : 13,591,643.70   (ALL orders, incl. ineligible)
freight   from raw item rows   : 2,251,909.54
payments  from raw rows        : 16,008,872.12
eligible merchandise as share  : 97.2767% of all item rows
no fan-out (orders unchanged)  : True



quality checks: 13 run, passed=True


## 3. Overall business performance

**Business question:** how did order volume, customer count and basket value move over
the observation window, and is growth coming from new or returning customers?

In [8]:
monthly = customer_metrics.monthly_trend(orders)
split = customer_metrics.monthly_new_vs_returning(orders)
fig = visualizations.new_vs_returning(split)
fig.update_layout(height=380)
fig.show()

In [9]:
recent = split.tail(6)
print(f"last 6 months: first-time buyers avg {recent['new_customers'].mean():,.0f}/month, "
      f"returning avg {recent['returning_customers'].mean():,.0f}/month")
print(f"returning share of monthly active customers (last 6m): "
      f"{(recent['returning_customers'].sum() / recent['total_customers'].sum()) * 100:.2f}%")
print(f"returning share over the whole window: "
      f"{(split['returning_customers'].sum() / split['total_customers'].sum()) * 100:.2f}%")

last 6 months: first-time buyers avg 6,306/month, returning avg 165/month
returning share of monthly active customers (last 6m): 2.55%
returning share over the whole window: 1.93%


**Observed.** Monthly activity is dominated by first-time buyers; returning customers
are a small, roughly flat band (see the printed shares above).

**Why it matters.** Growth in this window is acquisition-led. Any retention programme
is therefore fighting a strong compositional headwind — it must lift a very small
absolute base, which is why incremental lift must be measured rather than assumed.

**Alternative explanations.** The marketplace was scaling, so even a constant *per
customer* repeat propensity would show up as a declining returning share. Monthly
returning counts are also affected by cohort maturity at the end of the window.

In [10]:
fig = visualizations.trend_area(monthly, "n_orders", "Eligible orders per month", "Orders")
fig.update_layout(height=340)
fig.show()

## 4. Customer purchasing behaviour

**Business question:** how often does a customer buy, and how much of the base is
one-time-only?

In [11]:
cust = tables["customers"]
repeat = customer_metrics.repeat_purchase_stats(cust)
print(f"customers                 : {repeat['n_customers']:,}")
print(f"repeat customers (>=2)    : {repeat['n_repeat_customers']:,}")
print(f"one-time customers        : {repeat['n_one_time_customers']:,}")
print(f"repeat-purchase rate      : {repeat['repeat_purchase_rate']:.4%}")
print(f"orders per customer (mean): {repeat['orders_per_customer']:.4f}")
cust["n_orders"].value_counts().sort_index().head(8).rename("customers").to_frame()

customers                 : 93,358
repeat customers (>=2)    : 2,801
one-time customers        : 90,557
repeat-purchase rate      : 3.0003%
orders per customer (mean): 1.0334


,customers
n_orders,
1,90557
2,2573
3,181
4,28
5,9
6,5
7,3
9,1


In [12]:
intervals = customer_metrics.purchase_intervals(orders)
stats_int = customer_metrics.interval_summary(intervals)
pd.Series(stats_int).round(2)

n_intervals        3,120.0000
mean_days             79.1500
median_days           29.4600
p25_days               0.0100
p75_days             121.4700
p90_days             240.2800
p95_days             312.1800
share_within_30d       0.5000
share_within_90d       0.6900
dtype: float64

In [13]:
second = customer_metrics.time_to_second_purchase(orders)
fig = visualizations.time_to_second(second)
fig.update_layout(height=340)
fig.show()
print(f"repeaters observed: {len(second):,}")

repeaters observed: 2,801


**Observed.** See the printed interval statistics: the median gap between consecutive
orders is around a month, and roughly half of all observed gaps fall inside 30 days,
with a long tail out to ~240 days at the 90th percentile.

**Why it matters.** The repurchase window is short. Onboarding and post-purchase
communication should be front-loaded into the first weeks; long-horizon win-back
campaigns address a period in which almost nothing normally happens.

**Alternative explanations.** These gaps are measured among customers who *did* repeat,
so they describe repeaters, not the probability that any given customer repeats. The
long tail may reflect seasonal or occasion-driven categories rather than reactivation.

In [14]:
fig = visualizations.concentration_curve(cust.rename(columns={"merchandise_value": "monetary_value"}))
fig.update_layout(height=400)
fig.show()

In [15]:
conc = customer_metrics.concentration_stats(cust)
pd.Series({k: round(v, 4) if isinstance(v, float) else v for k, v in conc.items()})

total_merchandise_value   13,221,498.1100
gini                               0.5203
n_customers                   93,358.0000
top_1pct_share                     0.1147
top_5pct_share                     0.2913
top_10pct_share                    0.4110
top_20pct_share                    0.5662
top_50pct_share                    0.8331
bottom_50pct_share                 0.1669
dtype: float64

**Observed.** See the concentration statistics: the top 10% of customers hold roughly
two-fifths of historical merchandise value; the Gini coefficient is around 0.52.

**Why it matters.** Value is concentrated but *not* in a classic 80/20 shape — a large
share of revenue sits in the one-time-buyer tail. A VIP-only retention strategy would
address well under half of historical revenue.

**Alternative explanations.** Historical value mixes category mix and price level. A
single expensive purchase looks like "high value" without implying a valuable
relationship, and margin — which would separate the two — is absent from the dataset.

## 5. Cohort retention

**Business question:** of the customers who first bought in month X, what share bought
again in month X+n?

Definitions: cohort = month of first eligible order; month index = calendar months
since; retention = distinct active cohort members ÷ cohort size, for that specific
month. Month 0 is 100% by construction. **Months beyond the end of the dataset are not
created at all** — they are missing, never zero.

In [16]:
cohort = cohort_analysis.build_cohort_table(orders)
print("validation problems:", cohort_analysis.validate_cohort_table(cohort) or "none")
cohort_analysis.cohort_sizes(cohort).tail(6)

validation problems: none


,cohort_month,cohort_size,max_observable_index
17,2018-03-01,6774,5
18,2018-04-01,6582,4
19,2018-05-01,6506,3
20,2018-06-01,5878,2
21,2018-07-01,5949,1
22,2018-08-01,6144,0


In [17]:
matrix = cohort_analysis.cohort_matrix(cohort, max_index=12)
fig = visualizations.cohort_heatmap(matrix, max_index=12)
fig.update_layout(height=620)
fig.show()

In [18]:
pooled = cohort_analysis.weighted_retention_by_month(cohort, upto=12)
display = pooled.assign(retention_pct=lambda d: (d["retention_rate"] * 100).round(3))
display[["month_index", "n_active", "n_eligible", "n_cohorts", "retention_pct"]].head(13)

,month_index,n_active,n_eligible,n_cohorts,retention_pct
0,0,93358,93358,23,100.0000
1,1,421,87214,22,0.4830
2,2,273,81265,21,0.3360
3,3,192,75387,20,0.2550
4,4,176,68881,19,0.2560
5,5,140,62299,18,0.2250
6,6,129,55525,17,0.2320
7,7,102,49237,16,0.2070
8,8,86,42395,15,0.2030
9,9,60,37057,14,0.1620


In [19]:
kpis = cohort_analysis.retention_kpis(cohort, orders)
cum = cohort_analysis.cumulative_repeat_rate(cohort, orders, upto=12)
print(f"month-1 retention (pooled) : {kpis['month_1_retention_pooled']:.4%}")
print(f"month-2 retention (pooled) : {kpis['month_2_retention_pooled']:.4%}")
print(f"cohorts total / mature 12m : {kpis['n_cohorts']} / {kpis['n_cohorts_mature_12m']}")
print(f"median 12m cumulative      : {kpis['median_12m_cumulative_repeat_rate']:.4%}")
cum.head(6)

month-1 retention (pooled) : 0.4827%
month-2 retention (pooled) : 0.3359%
cohorts total / mature 12m : 23 / 11
median 12m cumulative      : 2.9051%


,cohort_month,cohort_size,max_observable_index,n_cumulative_repeaters,cumulative_repeat_rate
0,2016-09-01,1,23,0,0.0000
1,2016-10-01,262,22,3,0.0115
2,2016-12-01,1,20,1,1.0000
3,2017-01-01,717,19,20,0.0279
4,2017-02-01,1628,18,38,0.0233
5,2017-03-01,2503,17,72,0.0288


**Observed.** Specific-month retention is far below the 'ever repeated' rate — see the
printed values. The gap is not an error: specific-month retention counts *activity in
that month*, whereas the repeat rate counts anyone who ever returned.

**Why it matters.** Conflating the two is the classic cohort-reporting mistake. It is
also why retention dashboards that show "month-3 retention = 0.3%" next to "repeat
rate = 3%" look contradictory when they are both correct.

**Alternative explanations.** A retailer with an annual natural cycle would show low
monthly retention and high cumulative return. Here the cumulative 12-month figure is
also low, which is what makes the low-retention reading credible rather than an
artefact of granularity.

## 6. RFM segmentation

**Business question:** which customers are valuable and recent, which are lapsing, and
which are too new to judge?

Recency = days since last eligible order (to the analysis date). Frequency = distinct
eligible orders. Monetary = Σ merchandise value (**revenue-like, not profit**).
Thresholds are derived from observed repurchase gaps.

In [20]:
metrics = customer_metrics.build_customer_metrics(orders, tables["customers"])
print("thresholds:", {k: round(v, 1) for k, v in metrics.thresholds.items()})
seg = segmentation.run_segmentation(
    metrics.customers, method="rule_based",
    lapsing_days=metrics.thresholds["lapsing_days"],
    inactive_days=metrics.thresholds["inactive_days"])
profile = seg.profile
cols = ["segment", "n_customers", "share_customers", "n_orders", "monetary_value",
        "share_value", "avg_recency_days", "avg_frequency", "avg_order_value"]
profile[cols].round(3)

thresholds: {'lapsing_days': 121.5, 'inactive_days': 240.3, 'gap_p75_days': 121.5, 'gap_p90_days': 240.3, 'lapsing_floor': 90, 'inactive_floor': 180}


,segment,n_customers,share_customers,n_orders,monetary_value,share_value,avg_recency_days,avg_frequency,avg_order_value
0,High-Value Active Repeat,460,0.0050,1037,"190,635.8300",0.0140,62.1760,2.2540,183.8340
1,Active Repeat,385,0.0040,783,"42,201.3900",0.0030,62.4700,2.0340,53.8970
2,Lapsing Repeat,840,0.0090,1782,"215,221.2400",0.0160,181.8370,2.1210,120.7750
3,Recent One-Time Buyer,12427,0.1330,12427,"1,780,961.7700",0.1350,92.2750,1.0000,143.3140
4,Dormant One-Time Buyer,25529,0.2730,25529,"3,442,345.7400",0.2600,180.9440,1.0000,134.8410
5,Long Inactive,41746,0.4470,42949,"5,912,852.8700",0.4470,379.5750,1.0290,137.6710
6,New / Insufficient History,11971,0.1280,11971,"1,637,279.2700",0.1240,29.5200,1.0000,136.7700


In [21]:
fig = visualizations.segment_value_chart(profile)
fig.update_layout(height=420)
fig.show()

In [22]:
check = profile[["n_customers", "share_customers", "monetary_value", "share_value"]].sum()
print(check.round(4).to_string())
print(f"\ncustomers reconcile to base: {profile['n_customers'].sum() == len(metrics.customers)}")
print(f"value reconciles to base  : "
      f"{np.isclose(profile['monetary_value'].sum(), metrics.customers['monetary_value'].sum())}")

n_customers           93,358.0000
share_customers            1.0000
monetary_value    13,221,498.1100
share_value                1.0000

customers reconcile to base: True
value reconciles to base  : True


**Observed.** Segments reconcile exactly to the customer base and to total value (see
the checks above). The two repeat segments are small in absolute size even though they
carry value per customer well above average.

**Why it matters.** The "loyal core" is tiny. Retention strategy here cannot be a
VIP-protection programme — there is almost no VIP tier to protect. The larger
commercial question is how to move one-time buyers to a second order.

**Alternative explanations / limits.** Segments are a snapshot; boundaries are soft
(one day either side of a threshold changes the label); monetary value is revenue, not
profit; and these are proposed analytical categories, not Olist's own segmentation.

### K-means extension (optional)

Log-transformed, scaled RFM with k chosen by silhouette. Clusters group *shape*, not
commercial meaning — read each profile before naming it.

In [23]:
del fig
gc.collect()
km = segmentation.assign_kmeans_segments(metrics.customers.sample(20000, random_state=0))
print("best k:", km["best_k"], "silhouette:", round(km["best_silhouette"], 4))
km["silhouette"].round(4)

best k: 3 silhouette: 0.4196


,k,silhouette,inertia
0,3,0.4196,"27,790.0227"
1,4,0.3880,"17,952.4139"
2,5,0.3439,"14,914.2433"
3,6,0.3493,"12,721.2938"
4,7,0.3563,"11,124.6804"
5,8,0.3427,"9,989.1476"


In [24]:
km["profile"].round(3)

,cluster,n_customers,mean_recency,mean_frequency,mean_monetary,total_monetary,share_customers,share_value
0,0,15278,288.9450,1.0000,138.9350,"2,122,655.6900",0.7640,0.7490
1,1,4104,47.7100,1.0000,135.1390,"554,611.5700",0.2050,0.1960
2,2,618,209.4740,2.1180,256.1930,"158,327.1500",0.0310,0.0560


## 7. Service experience and later purchasing

**Business question:** do delivery problems and low review scores predict whether a
customer orders again?

Outcome = another eligible order within 90 days from the same customer. Orders whose
90-day window extends past the end of the dataset are **right-censored and excluded**.
Effect sizes (Cramér's V) matter more than p-values at this sample size.

In [25]:
reviews = service_analysis.review_summary(orders)
delivery = service_analysis.delivery_summary(orders)
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in reviews.items() if k != "distribution"})
print({k: round(v, 4) if isinstance(v, float) else v for k, v in delivery.items()})

{'n_reviews': 95832, 'n_orders': 96478, 'coverage': 0.9933, 'mean_score': 4.1559, 'median_score': 5.0, 'share_5': 0.5922, 'share_1_2': 0.1281}
{'n_delivered_with_date': 96470, 'median_delivery_days': 10.2175, 'p90_delivery_days': 23.0964, 'share_late': 0.0811, 'median_delay_days': -11.9481, 'median_late_delay_days': 5.8065}


In [26]:
outcome = service_analysis.add_repeat_outcome(orders, settings.SERVICE_REPEAT_WINDOW_DAYS,
                                              metrics.dates["as_of_date"])
outcome = service_analysis.late_delivery_bins(outcome)
bands = service_analysis.score_bands(outcome)
print(f"orders with a full 90-day window : {outcome['repeat_within_window'].notna().sum():,}")
print(f"orders censored (excluded)       : {outcome['censored'].sum():,}")
print(f"base rate of repeat in 90 days   : {outcome['repeat_within_window'].mean():.4%}")

orders with a full 90-day window : 77,737
orders censored (excluded)       : 18,741
base rate of repeat in 90 days   : 2.4789%


In [27]:
fig = visualizations.score_by_lateness(service_analysis.score_vs_lateness(outcome))
fig.update_layout(height=340)
fig.show()
service_analysis.score_vs_lateness(outcome).round(3)

,late_bin,n_orders,mean_review,share_low_score
0,Early,86713,4.2960,0.0920
1,On time,1450,4.1580,0.1100
2,1-3 days late,2636,3.7660,0.1910
3,4-7 days late,1773,2.3160,0.6130
4,8+ days late,3252,1.7270,0.7840


In [28]:
by_band = service_analysis.repeat_rate_by_group(bands, "score_band")
by_late = service_analysis.repeat_rate_by_group(outcome, "late_bin")
t_band = service_analysis.association_tests(bands, "score_band")
t_late = service_analysis.association_tests(outcome, "late_bin")
print("repeat rate by review band:")
print(by_band[["score_band", "n_orders", "repeat_rate", "ci_low", "ci_high"]].round(5).to_string(index=False))
print(f"  chi2 p={t_band['p_value']:.4g}  Cramers V={t_band['cramers_v']:.4f}"
      f"  ({service_analysis.interpret_effect_size(t_band['cramers_v'])})")
print("\nrepeat rate by lateness:")
print(by_late[["late_bin", "n_orders", "repeat_rate", "ci_low", "ci_high"]].round(5).to_string(index=False))
print(f"  chi2 p={t_late['p_value']:.4g}  Cramers V={t_late['cramers_v']:.4f}"
      f"  ({service_analysis.interpret_effect_size(t_late['cramers_v'])})")

repeat rate by review band:
score_band  n_orders  repeat_rate  ci_low  ci_high
       1-2     10460       0.0241  0.0213   0.0272
         3      6635       0.0240  0.0205   0.0279
         4     15378       0.0218  0.0196   0.0242
         5     44701       0.0260  0.0246   0.0275
  chi2 p=0.03011  Cramers V=0.0108  (negligible)

repeat rate by lateness:
     late_bin  n_orders  repeat_rate  ci_low  ci_high
        Early     69934       0.0251  0.0239   0.0262
      On time       995       0.0221  0.0146   0.0333
1-3 days late      2089       0.0244  0.0186   0.0320
4-7 days late      1584       0.0183  0.0128   0.0262
 8+ days late      3133       0.0230  0.0183   0.0288
  chi2 p=0.4473  Cramers V=0.0069  (negligible)


**Observed.** Delivery lateness is strongly and monotonically related to *satisfaction*
(mean review score collapses as lateness grows), but neither review band nor lateness
shows a practically meaningful association with *later repeat purchasing* — Cramér's V
is around 0.01 in both cases, and the confidence intervals overlap heavily.

**Why it matters.** This contradicts a widely assumed causal chain ("bad delivery →
bad review → customer lost"). Service quality clearly damages satisfaction; in this
dataset it does not measurably change whether the customer orders again. Any internal
business case that sells delivery improvement *as a retention lever* would be
overclaiming.

**Alternative explanations.** (1) The true effect may be small and within noise at a
~2–3% base rate. (2) Service failures may destroy high-intent customers while being
masked by the majority who were never going to return. (3) Review submission is
selective. (4) No randomisation exists, so confounding by region, category and
customer type is uncontrolled. The defensible conclusion is "not measurable here", not
"no effect exists".

## 8. Summary of what this notebook established

Run the cells above and read the printed values — the summary below is deliberately
qualitative so it cannot go stale:

1. **Repeat purchasing is rare** in this window; the base rate is the ceiling that any
   retention campaign operates under.
2. **When customers repeat, they do it quickly** — the median gap is short and about
   half of all repeat gaps fall inside 30 days.
3. **Specific-month retention is far lower than the ever-repeated rate**, and the two
   must never be quoted interchangeably.
4. **Value is concentrated but the tail still matters**; a top-decile-only strategy
   leaves most historical revenue untouched.
5. **The loyal core is small in absolute size**, so this is a conversion problem, not a
   VIP-protection problem.
6. **Service experience drives satisfaction but not measurably retention** — a result
   worth reporting precisely because it is inconvenient.

Everything remains **descriptive**. No campaign data exists, so no intervention can be
evaluated and no uplift attributed. See `reports/limitations.md`.